# Jim Simons Kantitatif Strateji — BIST

**Temel Bileşenler:**
1. **HMM** — Gizli Markov Modeli ile piyasa rejimi tespiti (Bull/Bear/Sideways)
2. **İstatistiki Arbitraj** — Korelasyonlu çift hisseler, mean reversion (Z-score)
3. **Anomali Taraması** — Geçmiş veriyle istatistiksel edge tespiti
4. **Kelly Kriteri** — Pozisyon boyutlandırma
5. **%51 Kuralı** — Küçük ama tekrarlanabilir edge'leri birleştirme

> Not: Günlük veriyle çalışır (Colab/yfinance). Gerçek Rentec sistemi milisaniye verisi kullanır.

In [ ]:
import subprocess, sys

def pip_install(pkg, label=None):
    label = label or pkg
    print(f"  {label} ...", end=" ")
    r = subprocess.run([sys.executable,"-m","pip","install","-q",pkg], capture_output=True, text=True)
    print("OK" if r.returncode==0 else f"HATA\n{r.stderr[-300:]}")

for pkg in [
    "pandas","numpy","requests","tqdm","openpyxl",
    "yfinance",
    "websocket-client",
    "websockets",
    "hmmlearn",
    "scikit-learn",
    "scipy",
    "statsmodels",
    "tradingview-screener",
]:
    pip_install(pkg)

# tvdatafeed -- GitHub (en guncel, BIST teknik veri icin en temiz kaynak)
pip_install(
    "git+https://github.com/rongardF/tvdatafeed.git",
    label="tvdatafeed (rongardF/GitHub)"
)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = "/content/drive/MyDrive/Simons_Quant"
    IN_COLAB = True
except Exception:
    ROOT = "/tmp/Simons_Quant"
    IN_COLAB = False

import os
os.makedirs(f"{ROOT}/cache", exist_ok=True)
os.makedirs(f"{ROOT}/raporlar", exist_ok=True)
print(f"\nRoot: {ROOT} | Colab: {IN_COLAB}")


In [ ]:
import warnings, time, pickle
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import yfinance as yf
from tqdm import tqdm
warnings.filterwarnings("ignore")

# ── Strateji Parametreleri ────────────────────────────────────────────────────
N_BARS            = 500    # Fiyat geçmişi (gün)
HMM_N_STATES      = 3      # Piyasa rejimleri: Bull / Sideways / Bear
HMM_LOOKBACK      = 252    # HMM eğitim penceresi (1 yıl)
ZSCORE_WINDOW     = 20     # Mean reversion Z-score penceresi
ZSCORE_ENTRY      = 2.0    # Giriş eşiği (|z| > bu değer)
ZSCORE_EXIT       = 0.5    # Çıkış eşiği (|z| < bu değer)
CORR_MIN          = 0.70   # Çift seçimi min korelasyon
COINT_PVALUE      = 0.05   # Koentegrasyon p-value eşiği
KELLY_FRACTION    = 0.25   # Kelly fraksiyonu (0.25 = quarter Kelly)
MAX_POSITION_PCT  = 0.10   # Tek pozisyon maks %10
CACHE_TTL_H       = 6      # Cache geçerlilik (saat)

print("Konfigürasyon yüklendi.")


In [ ]:
# -- Tum BIST Hisse Listesi ---------------------------------------------------

BIST_SABIT = [
    "ACSEL","ADEL","ADESE","ADGYO","AEFES","AFYON","AGESA","AGHOL","AGROT","AHGAZ",
    "AKBNK","AKFGY","AKGRT","AKMGY","AKSA","AKSEN","AKSGY","AKSUE","AKTIF","ALARK",
    "ALBRK","ALCAR","ALFAS","ALGYO","ALKIM","ALKLC","ALMAD","ALTNY","ALVES","ANELE",
    "ANGEN","ANHYT","ANSGR","ARASE","ARCLK","ARDYZ","ARENA","ARSAN","ASELS","ASGYO",
    "ASTOR","ATAGY","ATAKP","ATATP","ATEKS","ATLAS","AVGYO","AVHOL","AVOD","AVPGY",
    "AYCES","AYGAZ","AZTEK","BAGFS","BAKAB","BANVT","BARMA","BASCM","BASGZ","BAYRK",
    "BERA","BEYAZ","BFREN","BIMAS","BIOEN","BIZIM","BJKAS","BLCYT","BMEKS","BMSTL",
    "BNTAS","BORLS","BORSK","BOSSA","BRKO","BRKVY","BRMEN","BRSAN","BRYAT","BSOKE",
    "BTCIM","BUCIM","BURCE","BURVA","BVSAN","CCOLA","CELHA","CEMAS","CEMTS","CIMSA",
    "CLEBI","CMBTN","CMENT","COGAS","COSMO","CRDFA","CRFSA","CUSAN","CVKMD","CWENE",
    "DAGHL","DAGI","DAPGM","DARDL","DENGE","DERHL","DESA","DESPC","DEVA","DGATE",
    "DGNMO","DITAS","DMSAS","DNISI","DOAS","DOBUR","DOCO","DOGUB","DOHOL","DOSCH",
    "DPAZR","DRDGE","DTRND","DURDO","DYOBY","DZGYO","ECILC","ECZYT","EDIP","EFORC",
    "EGEEN","EGEPO","EGGUB","EGPRO","EGSER","EKGYO","EKIZ","EKOS","EKSUN","ELITE",
    "EMKEL","EMNIS","ENERY","ENKAI","ENSRI","EPLAS","ERBOS","ERCB","ERGL","EREGL",
    "ESCAR","ESCOM","ESEN","ETGRI","ETYAT","EUHOL","EUPWR","EUREN","EUYO","EVREN",
    "FADE","FENER","FMIZP","FONET","FORMT","FORTE","FROTO","FZLGY","GARAN","GARFA",
    "GEDIK","GEDZA","GENIL","GENTS","GEREL","GLYHO","GMTAS","GNDUZ","GOLTS","GOODY",
    "GOZDE","GRSEL","GRTHO","GSRAY","GUBRF","GUBRE","GULFA","GUMER","GUNAY","GWIND",
    "HALKB","HATEK","HDFGS","HEDEF","HEKTS","HKTM","HLGYO","HTTBT","HUBVC","HUNER",
    "HZNGY","ICBCT","IDGYO","IEDAS","IEYHO","IHEVA","IHGZT","IHLAS","IHLGM","IHMAD",
    "IHYAY","IMASM","INDES","INFO","INGRM","INTEM","INVEO","IPEKE","IPMAT",
    "ISCTR","ISFIN","ISGSY","ISGYO","ISYAT","ITTFK","JANTS","KAPLM","KAREL","KARSN",
    "KARTN","KERVN","KFEIN","KGYO","KRDMA","KRDMB","KRDMD","KLNMA","KLRHO","KMPUR",
    "KNFRT","KOCMT","KONTR","KONYA","KOPOL","KORDS","KOZAA","KOZAL","KRONT","KRPLS",
    "KSTUR","KTLEV","KURTL","KUYAS","KZBGY","LIDER","LIDFA","LINK","LKMNH","LOGO",
    "LRSHO","LYKHO","MAALT","MACKO","MAGEN","MAKIM","MANAS","MARKA","MARTI","MAVI",
    "MEDTR","MEGAP","MEPET","MERCN","MERIT","MERKO","METRO","METUR","MGROS","MIATK",
    "MIGRS","MMCAS","MNDRS","MNDTR","MOBTL","MOGAN","MPARK","MRGYO","MSGYO","MTRKS",
    "MZHLD","NATEN","NETAS","NIBAS","NIDDK","NTHOL","NTTUR","NUGYO","NUHCM","NXMGY",
    "ODAS","OFSYM","OLMIP","ONCSM","ONEN","ONRYT","ORGE","ORMA","OSTIM","OTKAR",
    "OTTO","OYAKC","OYLUM","OZGYO","OZKGY","OZRDN","OZSUB","PAGYO","PAMEL","PAPIL",
    "PARSN","PASEU","PCILT","PEGYO","PEKGY","PENGD","PENTA","PETKM","PETUN","PGSUS",
    "PINSU","PKART","PKENT","PLTUR","PNSUT","POLHO","POLTK","PORTK","PRDGS","PRZMA",
    "PSDTC","QNBFB","RYGYO","RGYAS","RHEAG","RNPOL","RODRG","ROYAL","RTALB","RUBNS",
    "SAFKR","SAHOL","SANEL","SANFM","SANKO","SARKY","SASA","SAYAS","SEKFK",
    "SEKUR","SELEC","SELGD","SELVA","SEYKM","SILVR","SISE","SKBNK","SKTAS","SMART",
    "SMRTG","SNGYO","SOKM","SONME","SRVGY","SUMAS","SUNTK","SURGY","SUWEN","SUZGT",
    "TARKM","TATGD","TAVHL","TBORG","TCELL","TDGYO","TEKTU","TETMT","THYAO","TIRE",
    "TKFEN","TKNSA","TLMAN","TMSN","TOASO","TRCAS","TRGYO","TRILC","TRNSK","TSPOR",
    "TSKB","TTKOM","TTRAK","TUCLK","TUKAS","TUMAS","TUPRS","TUREX","TURGG","TURSG",
    "UFUK","ULUFA","ULUSE","ULUUN","UNLU","USAK","USDAU","VAKBN","VAKFA","VAKFN",
    "VANGD","VBTYZ","VERUS","VESBE","VESTL","VKFYO","VKGYO","VRGYO","YBTAS","YATAS",
    "YGGYO","YKBNK","YKGYO","YKSLN","YONGA","YUNSA","ZEDUR","ZOREN","ZRGYO",
]

def get_bist_symbols():
    try:
        from tradingview_screener import Query, col as tvcol
        _, df = (
            Query()
            .select("name","close","volume","market_cap_basic")
            .where(
                tvcol("exchange").isin(["BIST"]),
                tvcol("type") == "stock",
            )
            .limit(600)
            .get_scanner_data()
        )
        syms = df["name"].str.replace("BIST:","").tolist()
        if len(syms) > 50:
            print(f"TradingView Screener: {len(syms)} hisse")
            return syms
    except Exception as e:
        print(f"Screener hatasi: {e}")
    print(f"Sabit liste: {len(BIST_SABIT)} hisse")
    return BIST_SABIT

SYMBOLS = get_bist_symbols()
print(f"Toplam: {len(SYMBOLS)} hisse taranacak")


In [ ]:
# -- Fiyat Verisi: SADECE tvdatafeed (rongardF/GitHub) -----------------------
import logging
logging.getLogger("tvDatafeed.main").setLevel(logging.CRITICAL)

from tvDatafeed import TvDatafeed, Interval

# Tek seferlik baglanti — anonim (login gerekmez)
print("tvdatafeed baglaniyor...")
TV = TvDatafeed()

# Baglanti testi
_test = TV.get_hist("THYAO", "BIST", Interval.in_daily, n_bars=5)
if _test is not None and len(_test) > 0:
    print("tvdatafeed: BAGLI")
else:
    print("UYARI: Veri alinamadi, exchange/symbol kontrol edin")

def _cache_path(sym):
    return Path(ROOT) / "cache" / f"{sym}.pkl"

def get_price(sym: str, n: int = N_BARS) -> pd.DataFrame:
    cp = _cache_path(sym)
    if cp.exists() and (time.time() - cp.stat().st_mtime) / 3600 < CACHE_TTL_H:
        with open(cp, "rb") as f:
            return pickle.load(f)

    for attempt in range(3):
        try:
            df = TV.get_hist(
                symbol=sym,
                exchange="BIST",
                interval=Interval.in_daily,
                n_bars=n + 50,
            )
            if df is not None and len(df) > 30:
                df.columns = [c.lower() for c in df.columns]
                df.index = pd.to_datetime(df.index)
                df = df.tail(n).copy()
                df = df[df["close"] > 0].dropna(subset=["close"])
                if len(df) > 30:
                    with open(cp, "wb") as f:
                        pickle.dump(df, f)
                    return df
        except Exception:
            time.sleep(2 ** attempt)

    return pd.DataFrame()

# -- Benchmark: BIST-100 (TVC exchange) --------------------------------------
print("\nBenchmark (XU100) cekiliyor...")
BENCH = pd.DataFrame()
for attempt in range(3):
    try:
        _b = TV.get_hist("XU100", "TVC", Interval.in_daily, n_bars=700)
        if _b is not None and len(_b) > 100:
            _b.columns = [c.lower() for c in _b.columns]
            _b.index = pd.to_datetime(_b.index)
            BENCH = _b
            break
    except Exception:
        time.sleep(2 ** attempt)

if not BENCH.empty:
    BENCHMARK_CLOSE = BENCH["close"]
    print(f"Benchmark OK: {len(BENCH)} bar, son: {BENCH['close'].iloc[-1]:.0f}")
else:
    BENCHMARK_CLOSE = None
    print("UYARI: Benchmark alinamadi.")

# -- Test --------------------------------------------------------------------
print("\nTHYAO test...")
_t = get_price("THYAO")
if not _t.empty:
    print(f"OK -- {len(_t)} bar, son fiyat: {_t['close'].iloc[-1]:.2f} TL")
else:
    print("HATA: Veri alinamadi")


In [ ]:
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler

def fit_hmm(close: pd.Series, n_states: int = HMM_N_STATES) -> dict:
    """
    Gizli Markov Modeli ile piyasa rejimini tespit eder.
    Özellikler: günlük getiri, volatilite, hacim değişimi
    Çıktı: her gün için rejim etiketi (0=Bear, 1=Sideways, 2=Bull)
    """
    if len(close) < HMM_LOOKBACK:
        return {"regime": None, "states": [], "probs": []}

    # Özellik matrisi
    ret      = close.pct_change().fillna(0)
    vol      = ret.rolling(5).std().fillna(0)
    ret_5d   = close.pct_change(5).fillna(0)

    X = np.column_stack([ret.values, vol.values, ret_5d.values])
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # HMM eğitimi
    model = GaussianHMM(
        n_components=n_states,
        covariance_type="full",
        n_iter=200,
        random_state=42,
    )
    try:
        model.fit(X_scaled[-HMM_LOOKBACK:])
        hidden_states = model.predict(X_scaled)
        state_probs   = model.predict_proba(X_scaled)
    except Exception as e:
        return {"regime": None, "states": [], "probs": []}

    # Rejim etiketleme: ortalama getiriye göre sırala
    state_returns = {}
    for s in range(n_states):
        mask = hidden_states == s
        state_returns[s] = ret.values[mask].mean() if mask.sum() > 0 else 0

    # Bull > Sideways > Bear sıralaması
    sorted_states = sorted(state_returns, key=lambda s: state_returns[s])
    label_map = {sorted_states[0]: "Bear", sorted_states[1]: "Sideways", sorted_states[2]: "Bull"}
    if n_states == 2:
        label_map = {sorted_states[0]: "Bear", sorted_states[1]: "Bull"}

    labeled = [label_map[s] for s in hidden_states]
    current_regime  = labeled[-1]
    current_probs   = {label_map[s]: float(state_probs[-1][s]) for s in range(n_states)}

    return {
        "regime":   current_regime,
        "prob":     current_probs,
        "history":  labeled,
        "last_5":   labeled[-5:],
    }

# Benchmark'a uygula
print("Piyasa rejimi analizi (BIST-100)...")
if not BENCH.empty:
    hmm_result = fit_hmm(BENCH["close"])
    print(f"  Güncel rejim  : {hmm_result['regime']}")
    print(f"  Son 5 gün     : {hmm_result['last_5']}")
    print(f"  Olasılıklar   : {hmm_result['prob']}")
    MARKET_REGIME = hmm_result["regime"]
else:
    MARKET_REGIME = "Unknown"
    print("Benchmark yok, rejim belirlenemedi.")


In [ ]:
from statsmodels.tsa.stattools import coint
from itertools import combinations

def find_cointegrated_pairs(symbols: list, price_dict: dict) -> list:
    """Koentegre cift hisseleri bulur (mean reversion icin ideal)."""
    valid = {s: price_dict[s]["close"] for s in symbols
             if s in price_dict and len(price_dict[s]) > 100}

    pairs = []
    syms  = list(valid.keys())
    total = len(list(combinations(syms, 2)))
    print(f"{len(syms)} hisse, {total} cift test ediliyor...")

    for s1, s2 in tqdm(combinations(syms, 2), total=total, desc="Cift tarama"):
        c1, c2 = valid[s1].align(valid[s2], join="inner")
        if len(c1) < 60:
            continue
        corr = c1.corr(c2)
        if abs(corr) < CORR_MIN:
            continue
        try:
            _, pval, _ = coint(c1.values, c2.values)
        except Exception:
            continue
        if pval < COINT_PVALUE:
            pairs.append({
                "s1": s1, "s2": s2,
                "corr": round(corr, 3),
                "coint_pval": round(pval, 4),
            })

    pairs.sort(key=lambda x: x["coint_pval"])
    print(f"\nBulunan koentegre cift: {len(pairs)}")
    return pairs


def calc_spread_zscore(c1: pd.Series, c2: pd.Series, window: int = ZSCORE_WINDOW) -> pd.Series:
    """Spread Z-score serisi."""
    c1, c2 = c1.align(c2, join="inner")
    ratio  = c1 / c2
    mean   = ratio.rolling(window).mean()
    std    = ratio.rolling(window).std()
    return (ratio - mean) / (std + 1e-9)


def get_pair_signal(s1: str, s2: str, price_dict: dict) -> dict:
    """Bir cift icin guncel mean reversion sinyali."""
    if s1 not in price_dict or s2 not in price_dict:
        return {}
    c1 = price_dict[s1]["close"]
    c2 = price_dict[s2]["close"]
    z  = calc_spread_zscore(c1, c2)
    if z.empty or pd.isna(z.iloc[-1]):
        return {}

    current_z = float(z.iloc[-1])
    if current_z > ZSCORE_ENTRY:
        action = f"LONG {s2} / SHORT {s1}"
    elif current_z < -ZSCORE_ENTRY:
        action = f"LONG {s1} / SHORT {s2}"
    elif abs(current_z) < ZSCORE_EXIT:
        action = "KAPAT (ortaya donus)"
    else:
        action = "Bekle"

    return {
        "s1": s1, "s2": s2,
        "zscore": round(current_z, 3),
        "action": action,
        "signal": abs(current_z) > ZSCORE_ENTRY,
    }

print("Pairs trading fonksiyonlari hazir.")


In [ ]:
from scipy import stats

def calc_edge_score(close: pd.Series, volume: pd.Series) -> dict:
    if len(close) < 60:
        return {"score": 0, "max_score": 6}

    # NaN temizligi
    close  = close.ffill().bfill()
    volume = volume.fillna(0)
    volume = volume.replace(0, volume[volume > 0].median() if (volume > 0).any() else 1)

    ret = close.pct_change().dropna()
    if len(ret) < 20:
        return {"score": 0, "max_score": 6}

    scores = {}

    # E1: Momentum anomalisi
    ret_5d = close.pct_change(5).dropna()
    if len(ret_5d) > 10:
        z_mom = (float(ret_5d.iloc[-1]) - float(ret_5d.mean())) / (float(ret_5d.std()) + 1e-9)
        scores["momentum_z"] = round(z_mom, 3)
        scores["E1_momentum"] = 1 if 0.8 < z_mom < 3.5 else 0
    else:
        scores["E1_momentum"] = 0

    # E2: Volatilite sikismasi
    try:
        ma20  = close.rolling(20).mean()
        std20 = close.rolling(20).std()
        ratio = (std20 / (ma20 + 1e-9)).dropna()
        if len(ratio) > 5:
            bb_pct = float(stats.percentileofscore(ratio.values, float(ratio.iloc[-1]))) / 100
            scores["bb_width_pct"] = round(bb_pct, 3)
            scores["E2_squeeze"] = 1 if bb_pct < 0.30 else 0
        else:
            scores["E2_squeeze"] = 0
    except Exception:
        scores["E2_squeeze"] = 0

    # E3: Hacim anomalisi (OBV egimi)
    try:
        obv = (np.sign(close.diff()) * volume).cumsum()
        n   = min(20, len(obv))
        slope = float(np.polyfit(range(n), obv.iloc[-n:].values, 1)[0])
        scores["obv_slope"] = round(slope, 0)
        scores["E3_volume"] = 1 if slope > 0 else 0
    except Exception:
        scores["E3_volume"] = 0

    # E4: Mean reversion potansiyeli
    try:
        delta = close.diff()
        gain  = delta.where(delta > 0, 0).rolling(14, min_periods=5).mean()
        loss  = (-delta).where(-delta > 0, 0).rolling(14, min_periods=5).mean()
        rsi   = 100 - 100 / (1 + gain / (loss + 1e-9))
        rsi_now  = float(rsi.iloc[-1])
        rsi_prev = float(rsi.iloc[-4])
        scores["rsi"] = round(rsi_now, 1)
        scores["E4_mean_rev"] = 1 if (28 < rsi_now < 52 and rsi_now > rsi_prev) else 0
    except Exception:
        scores["rsi"] = 50
        scores["E4_mean_rev"] = 0

    # E5: Kisa vadeli donus pattern (3 gun dusus + bugun pozitif)
    try:
        last4 = ret.iloc[-4:].values
        scores["E5_pattern"] = 1 if (all(r < 0 for r in last4[:3]) and last4[-1] > 0) else 0
    except Exception:
        scores["E5_pattern"] = 0

    # E6: Regime uyumu
    try:
        if BENCHMARK_CLOSE is not None and len(BENCHMARK_CLOSE) > 5:
            bench_5d = float(BENCHMARK_CLOSE.pct_change(5).iloc[-1])
            stock_5d = float(close.pct_change(5).iloc[-1])
            scores["E6_regime"] = 1 if (bench_5d > 0 and stock_5d > bench_5d) else 0
        else:
            scores["E6_regime"] = 0
    except Exception:
        scores["E6_regime"] = 0

    edge_keys = [k for k in scores if k.startswith("E")]
    scores["score"]     = sum(scores[k] for k in edge_keys)
    scores["max_score"] = len(edge_keys)
    return scores

print("Edge tarama fonksiyonu hazir.")


In [ ]:
def kelly_position(win_rate: float, avg_win: float, avg_loss: float,
                   portfolio_size: float = 1.0) -> dict:
    """
    Modifiye Kelly Kriteri ile optimal pozisyon boyutu.
    win_rate : kazanan işlem oranı (örn 0.52)
    avg_win  : ortalama kazanç (örn 0.03 = %3)
    avg_loss : ortalama kayıp (pozitif sayı, örn 0.02 = %2)
    """
    if avg_loss <= 0 or avg_win <= 0:
        return {"kelly_pct": 0, "position_size": 0}

    b = avg_win / avg_loss   # Kazanç/Kayıp oranı
    p = win_rate
    q = 1 - p

    kelly_full = (p * b - q) / b   # Tam Kelly
    kelly_frac = kelly_full * KELLY_FRACTION  # Quarter Kelly (daha güvenli)

    # Üst sınır
    kelly_frac = min(kelly_frac, MAX_POSITION_PCT)
    kelly_frac = max(kelly_frac, 0)

    return {
        "kelly_full_pct": round(kelly_full * 100, 2),
        "kelly_frac_pct": round(kelly_frac * 100, 2),
        "position_size":  round(kelly_frac * portfolio_size, 2),
        "win_rate":        win_rate,
        "payoff_ratio":    round(b, 2),
    }


def backtest_win_rate(close: pd.Series, entry_signal: pd.Series,
                      hold_days: int = 5) -> dict:
    """
    Geçmiş veriden win rate ve avg win/loss hesaplar.
    entry_signal: True olan günlerde alım yapıldığını varsayar.
    """
    wins, losses = [], []
    signal_dates = entry_signal[entry_signal].index

    for dt in signal_dates:
        try:
            idx = close.index.get_loc(dt)
            if idx + hold_days >= len(close):
                continue
            entry = close.iloc[idx]
            exit_ = close.iloc[idx + hold_days]
            ret   = (exit_ - entry) / entry
            if ret > 0:
                wins.append(ret)
            else:
                losses.append(abs(ret))
        except Exception:
            continue

    if not wins and not losses:
        return {"win_rate": 0.5, "avg_win": 0.02, "avg_loss": 0.02, "n_trades": 0}

    total  = len(wins) + len(losses)
    return {
        "win_rate": round(len(wins)/total, 3) if total > 0 else 0.5,
        "avg_win":  round(np.mean(wins), 4)  if wins   else 0.02,
        "avg_loss": round(np.mean(losses),4) if losses else 0.02,
        "n_trades": total,
    }

print("Kelly pozisyon boyutlandırma hazır.")


In [ ]:
# -- Tum Hisselerin Verisini Indir (tvdatafeed) ------------------------------
print(f"Veri indirme basliyor: {len(SYMBOLS)} hisse (tvdatafeed)")
print("-" * 50)

PRICE_DATA = {}
failed = []

for sym in tqdm(SYMBOLS, desc="Indiriliyor"):
    df = get_price(sym)
    if not df.empty:
        PRICE_DATA[sym] = df
    else:
        failed.append(sym)
    time.sleep(0.1)  # Rate limit

print(f"\nBasarili : {len(PRICE_DATA)}/{len(SYMBOLS)}")
print(f"Basarisiz: {len(failed)}")
if failed:
    print(f"  {failed[:20]}")


In [ ]:
# -- Pattern Madenciligi: gecmis buyuk yukselisleri ogren ---------------------
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

MIN_RISE_PCT   = 0.30   # Dusuruldu: %30 (BIST icin daha gercekci)
RISE_WINDOW    = 60     # Arttirildi: 60 gunde bu yukselis
PRE_RISE_DAYS  = 20     # Yukselis oncesi feature penceresi
MIN_SIMILARITY = 0.65   # Benzerlik esigi (biraz dusuk)


def extract_features(close, volume, start_idx):
    end_idx = start_idx + PRE_RISE_DAYS
    if end_idx > len(close):
        return None
    c = close.iloc[start_idx:end_idx].copy()
    v = volume.iloc[start_idx:end_idx].copy()

    # NaN / sifir temizligi
    c = c.fillna(method="ffill").fillna(method="bfill")
    v = v.fillna(0).replace(0, 1)  # Sifir hacmi 1'e esitle

    if c.std() < 1e-9 or len(c) < PRE_RISE_DAYS:
        return None

    c_mean = c.mean()
    if c_mean == 0:
        return None

    ret   = c.pct_change().fillna(0)
    v_rel = v / (v.mean() + 1e-9)
    obv   = (np.sign(c.diff().fillna(0)) * v).cumsum()

    delta = c.diff().fillna(0)
    gain  = delta.where(delta > 0, 0).rolling(5, min_periods=1).mean()
    loss  = (-delta).where(-delta > 0, 0).rolling(5, min_periods=1).mean()
    rs    = float(gain.iloc[-1]) / (float(loss.iloc[-1]) + 1e-9)
    rsi_n = (100 - 100 / (1 + rs)) / 100

    try:
        vol_slope = float(np.polyfit(range(PRE_RISE_DAYS), v_rel.values, 1)[0])
        obv_slope = float(np.polyfit(range(PRE_RISE_DAYS), obv.values, 1)[0])
        short_vol = float(c.rolling(5, min_periods=1).std().iloc[-1])
        long_vol  = float(c.rolling(PRE_RISE_DAYS, min_periods=5).std().iloc[-1])
        vol_ratio = short_vol / (long_vol + 1e-9)
    except Exception:
        return None

    price_range = (float(c.max()) - float(c.min())) / (c_mean + 1e-9)
    band_pos    = (float(c.iloc[-1]) - float(c.min())) / (float(c.max()) - float(c.min()) + 1e-9)
    c_slope     = float(c.iloc[-5:].mean()) / (float(c.iloc[:5].mean()) + 1e-9) - 1
    mom_diff    = float(ret.iloc[-5:].mean()) - float(ret.iloc[:5].mean())

    feat = np.array([
        float(ret.mean()), float(ret.std()), price_range, band_pos,
        c_slope, mom_diff, float(v_rel.mean()), vol_slope,
        float(v_rel.iloc[-5:].mean() / (v_rel.iloc[:5].mean() + 1e-9)),
        obv_slope, rsi_n, vol_ratio,
    ])

    return feat if not np.any(np.isnan(feat)) and not np.any(np.isinf(feat)) else None


def mine_patterns(price_dict):
    all_features, all_labels = [], []
    skipped_short = skipped_feat = found = 0

    print(f"Pattern madenciligi: {len(price_dict)} hisse")
    print(f"Kriter: {RISE_WINDOW} gunde >%{int(MIN_RISE_PCT*100)} yukselis")

    for sym, df in tqdm(price_dict.items(), desc="Pattern tarama"):
        close  = df["close"].reset_index(drop=True)
        volume = (df["volume"].reset_index(drop=True)
                  if "volume" in df.columns
                  else pd.Series(np.ones(len(close))))
        volume = volume.fillna(1).replace(0, 1)
        dates  = df.index

        min_len = PRE_RISE_DAYS + RISE_WINDOW + 5
        if len(close) < min_len:
            skipped_short += 1
            continue

        for i in range(PRE_RISE_DAYS, len(close) - RISE_WINDOW):
            entry = float(close.iloc[i])
            if entry <= 0 or np.isnan(entry):
                continue
            future_max = float(close.iloc[i: i + RISE_WINDOW].max())
            rise = (future_max - entry) / (entry + 1e-9)
            if rise < MIN_RISE_PCT:
                continue
            feat = extract_features(close, volume, i - PRE_RISE_DAYS)
            if feat is None:
                skipped_feat += 1
                continue
            all_features.append(feat)
            all_labels.append({
                "sym":      sym,
                "rise_pct": round(rise * 100, 1),
                "date":     str(dates[i]) if i < len(dates) else "?",
            })
            found += 1

    print(f"  Kisa veri atlanan : {skipped_short}")
    print(f"  Feature hatasi    : {skipped_feat}")
    print(f"  Bulunan pattern   : {found}")

    if not all_features:
        print("UYARI: Pattern bulunamadi!")
        return np.array([]), [], None

    matrix = np.array(all_features)
    scaler = StandardScaler()
    matrix_scaled = scaler.fit_transform(matrix)
    rises = [l["rise_pct"] for l in all_labels]
    print(f"Ort. yukselis: %{np.mean(rises):.1f}  |  Max: %{max(rises):.1f}")
    return matrix_scaled, all_labels, scaler


PATTERN_MATRIX, PATTERN_LABELS, PATTERN_SCALER = mine_patterns(PRICE_DATA)
print(f"\nPattern kutuphanesi: {len(PATTERN_LABELS)} pattern")


In [ ]:
# -- Guncel hisseleri pattern kutuphanesiyle karsilastir ----------------------

def scan_by_similarity(price_dict, pattern_matrix, pattern_labels, scaler):
    if len(pattern_matrix) == 0:
        print("Pattern kutuphanesi bos!")
        return []
    results = []
    print(f"Benzerlik taramasi: {len(price_dict)} hisse...")
    for sym, df in tqdm(price_dict.items(), desc="Benzerlik tarama"):
        close  = df["close"].reset_index(drop=True)
        volume = (df["volume"].reset_index(drop=True)
                  if "volume" in df.columns
                  else pd.Series(np.ones(len(close))))
        if len(close) < PRE_RISE_DAYS + 5:
            continue
        feat = extract_features(close, volume, len(close) - PRE_RISE_DAYS)
        if feat is None:
            continue
        feat_scaled     = scaler.transform(feat.reshape(1, -1))
        sims            = cosine_similarity(feat_scaled, pattern_matrix)[0]
        top_idx         = np.argsort(sims)[-10:][::-1]
        top_sim         = float(sims[top_idx[0]])
        avg_top5        = float(sims[top_idx[:5]].mean())
        if avg_top5 < MIN_SIMILARITY:
            continue
        top_patterns    = [pattern_labels[i] for i in top_idx[:10]]
        avg_rise        = float(np.mean([p["rise_pct"] for p in top_patterns]))
        max_rise        = float(max(p["rise_pct"] for p in top_patterns))
        benzer          = ", ".join(list(set(p["sym"] for p in top_patterns[:5])))
        results.append({
            "hisse":            sym,
            "fiyat":            round(float(df["close"].iloc[-1]), 2),
            "benzerlik_skoru":  round(avg_top5, 3),
            "max_benzerlik":    round(top_sim, 3),
            "tahmini_yukselis": round(avg_rise, 1),
            "max_yukselis":     round(max_rise, 1),
            "benzer_hisseler":  benzer,
        })
    results.sort(key=lambda x: (x["benzerlik_skoru"], x["tahmini_yukselis"]), reverse=True)
    return results


SIMILARITY_RESULTS = scan_by_similarity(
    PRICE_DATA, PATTERN_MATRIX, PATTERN_LABELS, PATTERN_SCALER
)
print(f"\nBenzer pattern bulunan hisse: {len(SIMILARITY_RESULTS)}")
if SIMILARITY_RESULTS:
    df_sim = pd.DataFrame(SIMILARITY_RESULTS)
    print("\n-- EN YUKSEK BENZERLIK SKORLARI -----------------------------------")
    print(df_sim[["hisse","fiyat","benzerlik_skoru","tahmini_yukselis",
                  "max_yukselis","benzer_hisseler"]].head(20).to_string(index=False))


In [ ]:
# ── Edge + HMM Taraması ───────────────────────────────────────────────────────
print(f"Edge taraması başlıyor... Piyasa rejimi: {MARKET_REGIME}")
print("-" * 55)

edge_results = []

for sym, df in tqdm(PRICE_DATA.items(), desc="Edge tarama"):
    try:
        close  = df["close"]
        volume = df["volume"] if "volume" in df.columns else pd.Series(dtype=float)

        if volume.empty:
            volume = pd.Series(np.ones(len(close)), index=close.index)

        scores = calc_edge_score(close, volume)
        if scores["score"] < 3:
            continue

        # Bireysel HMM rejimi
        hmm = fit_hmm(close, n_states=2)  # Hızlı: sadece 2 durum
        stock_regime = hmm.get("regime", "?")

        # Backtest win rate (son 252 bar, 5 gün tutma)
        signal_series = pd.Series(False, index=close.index)
        # Basit giriş sinyali: RSI < 45 ve OBV pozitif
        ret   = close.pct_change().fillna(0)
        delta = close.diff()
        gain  = delta.where(delta>0,0).rolling(14).mean()
        loss  = (-delta).where(delta<0,0).rolling(14).mean()
        rsi_s = 100 - 100/(1 + gain/(loss+1e-9))
        obv_s = (np.sign(close.diff()) * volume).cumsum()
        obv_slope_s = obv_s.rolling(10).apply(lambda x: np.polyfit(range(len(x)),x,1)[0], raw=True)

        signal_series = (rsi_s < 48) & (obv_slope_s > 0)
        bt = backtest_win_rate(close.iloc[-252:], signal_series.iloc[-252:], hold_days=5)

        # Kelly
        kelly = kelly_position(
            win_rate=bt["win_rate"],
            avg_win=bt["avg_win"],
            avg_loss=bt["avg_loss"],
        )

        edge_results.append({
            "hisse":           sym,
            "fiyat":           round(float(close.iloc[-1]), 2),
            "edge_skor":       f"{scores['score']}/{scores['max_score']}",
            "piyasa_rejim":    MARKET_REGIME,
            "hisse_rejim":     stock_regime,
            "RSI":             scores.get("rsi", "-"),
            "E1_momentum":     "✓" if scores.get("E1_momentum") else "✗",
            "E2_squeeze":      "✓" if scores.get("E2_squeeze")  else "✗",
            "E3_hacim":        "✓" if scores.get("E3_volume")   else "✗",
            "E4_mean_rev":     "✓" if scores.get("E4_mean_rev") else "✗",
            "E5_patern":       "✓" if scores.get("E5_pattern")  else "✗",
            "E6_rejim":        "✓" if scores.get("E6_regime")   else "✗",
            "win_rate_pct":    round(bt["win_rate"]*100, 1),
            "n_islem":         bt["n_trades"],
            "kelly_pct":       kelly["kelly_frac_pct"],
            "payoff_oran":     kelly["payoff_ratio"],
        })
    except Exception:
        continue

print(f"\nEdge sinyali veren hisse: {len(edge_results)}")


In [ ]:
# ── Koentegre Çift Tarama (Mean Reversion) ───────────────────────────────────
# Sadece en likit 50 hisseyi çift testine sok (kombinasyon sayısı makul kalsın)
liquid_syms = list(PRICE_DATA.keys())[:50]
print(f"Çift taraması: {liquid_syms[:50]} hissede...")

PAIRS = find_cointegrated_pairs(liquid_syms, PRICE_DATA)

# Aktif çift sinyalleri
pair_signals = []
for p in PAIRS[:30]:  # En iyi 30 çift
    sig = get_pair_signal(p["s1"], p["s2"], PRICE_DATA)
    if sig.get("signal"):
        pair_signals.append({**p, **sig})

print(f"Aktif çift sinyali: {len(pair_signals)}")
if pair_signals:
    df_pairs = pd.DataFrame(pair_signals)
    print(df_pairs[["s1","s2","corr","coint_pval","zscore","action"]].to_string(index=False))


In [ ]:
# -- Nihai Rapor --------------------------------------------------------------
now_str = datetime.now().strftime("%d.%m.%Y %H:%M")
sep = "=" * 65
print(f"\n{sep}")
print(f"  JIM SIMONS KANTITATIF TARAMA -- {now_str}")
print(f"{sep}")
print(f"  Piyasa rejimi       : {MARKET_REGIME}")
print(f"  Taranan hisse       : {len(PRICE_DATA)}")
print(f"  Pattern kutuphanesi : {len(PATTERN_LABELS)} gecmis pattern")
print(f"  Benzerlik sinyali   : {len(SIMILARITY_RESULTS)}")
print(f"  Edge sinyali        : {len(edge_results)}")
print(f"  Aktif cift          : {len(pair_signals)}")
print(f"{sep}\n")

ts = datetime.now().strftime("%Y%m%d_%H%M")
xl = f"{ROOT}/raporlar/simons_quant_{ts}.xlsx"

with pd.ExcelWriter(xl, engine="openpyxl") as writer:
    if SIMILARITY_RESULTS:
        df_sim = pd.DataFrame(SIMILARITY_RESULTS).sort_values("benzerlik_skoru", ascending=False)
        df_sim.to_excel(writer, sheet_name="Pattern_Benzerligi", index=False)
        print("-- PATTERN BENZERLIGI (Tahta Yapici Izi) ----------------------")
        print(df_sim[["hisse","fiyat","benzerlik_skoru","tahmini_yukselis",
                       "max_yukselis","benzer_hisseler"]].head(20).to_string(index=False))
    if edge_results:
        df_edge = pd.DataFrame(edge_results).sort_values(
            ["edge_skor","win_rate_pct"], ascending=False)
        df_edge.to_excel(writer, sheet_name="Edge_Sinyaller", index=False)
        print("\n-- EDGE SINYALLERI --------------------------------------------")
        print(df_edge.head(20).to_string(index=False))
    if pair_signals:
        pd.DataFrame(pair_signals).to_excel(writer, sheet_name="Cift_Islemler", index=False)
    if SIMILARITY_RESULTS and edge_results:
        sim_set  = {r["hisse"] for r in SIMILARITY_RESULTS}
        edge_set = {r["hisse"] for r in edge_results}
        kesisim  = sim_set & edge_set
        if kesisim:
            star = "*" * 55
            print(f"\n{star}")
            print(f"  KESISIM: Her iki yontemde sinyal veren hisseler:")
            print(f"  {', '.join(sorted(kesisim))}")
            print(f"{star}")
            df_sim[df_sim["hisse"].isin(kesisim)].to_excel(
                writer, sheet_name="Kesisim_Guclu", index=False)

print(f"\nExcel: {xl}")


In [ ]:
# ── Tekil Hisse Detay Analizi ─────────────────────────────────────────────────
def full_analysis(sym: str, portfolio_tl: float = 100_000):
    df = PRICE_DATA.get(sym, pd.DataFrame())
    if df.empty:
        df = get_price(sym)
    if df is None or df.empty:
        print(f"{sym}: veri yok")
        return

    close  = df["close"]
    volume = df.get("volume", pd.Series(np.ones(len(close)), index=close.index))

    print(f"\n{'='*55}")
    print(f"  {sym} — SIMONS KANTİTATİF ANALİZ")
    print(f"{'='*55}")
    print(f"  Fiyat      : {close.iloc[-1]:.2f} TL")
    print(f"  1G         : {(close.iloc[-1]/close.iloc[-2]-1)*100:+.2f}%")
    print(f"  5G         : {(close.iloc[-1]/close.iloc[-5]-1)*100:+.2f}%")
    print(f"  20G        : {(close.iloc[-1]/close.iloc[-20]-1)*100:+.2f}%")

    print("\n── HMM Rejim Analizi ──────────────────────────")
    hmm = fit_hmm(close)
    print(f"  Güncel rejim : {hmm['regime']}")
    print(f"  Son 5 gün    : {hmm['last_5']}")
    probs = hmm.get("prob", {})
    for k, v in probs.items():
        print(f"  {k:10}: %{v*100:.1f}")

    print("\n── Edge Skorları ──────────────────────────────")
    sc = calc_edge_score(close, volume)
    for k, v in sc.items():
        if k.startswith("E"):
            print(f"  {k}: {'✓' if v else '✗'}")
    print(f"  TOPLAM: {sc['score']}/{sc['max_score']}")

    print("\n── Kelly Pozisyon ─────────────────────────────")
    delta = close.diff()
    gain  = delta.where(delta>0,0).rolling(14).mean()
    loss  = (-delta).where(delta<0,0).rolling(14).mean()
    rsi_s = 100 - 100/(1 + gain/(loss+1e-9))
    obv_s = (np.sign(close.diff()) * volume).cumsum()
    obv_slope_s = obv_s.rolling(10).apply(lambda x: np.polyfit(range(len(x)),x,1)[0], raw=True)
    sig = (rsi_s < 48) & (obv_slope_s > 0)
    bt  = backtest_win_rate(close, sig, hold_days=5)
    k   = kelly_position(bt["win_rate"], bt["avg_win"], bt["avg_loss"], portfolio_tl)
    print(f"  Geçmiş işlem : {bt['n_trades']} adet")
    print(f"  Win rate     : %{bt['win_rate']*100:.1f}")
    print(f"  Payoff oranı : {k['payoff_ratio']:.2f}x")
    print(f"  Tam Kelly    : %{k['kelly_full_pct']:.2f}")
    print(f"  Quarter Kelly: %{k['kelly_frac_pct']:.2f}")
    print(f"  Pozisyon TL  : {k['position_size']:,.0f} TL")
    print()

# Örnek
full_analysis("THYAO", portfolio_tl=100_000)


In [ ]:
# Analiz etmek istediğin hisseyi buraya yaz
HISSE = "GARAN"
PORTFOY_TL = 100_000

full_analysis(HISSE, portfolio_tl=PORTFOY_TL)
